In [68]:
import polars as pl
from utils import getConnection
from utils import cleanContracts
from utils import readCSV

pl.Config.set_tbl_rows(100) # Show up to 100 rows

con, dataset_path = getConnection() # Create the duckDB connection
readCSV(con, dataset_path) # Read CSV and create temp contracts table
cleanContracts(con) # Clean the data before we begin our financial analysis

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

# Create temp table where one row represents the latest state of the contract

In [ ]:
# This simplifies later queries 
con.sql("""CREATE TEMP TABLE contracts_latest AS
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY reference_number
               ORDER BY contract_date DESC
           ) AS rn
    FROM contracts_clean
) t
WHERE rn = 1;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

# Top 10 vendors by contract value

In [70]:
with pl.Config(float_precision=2, thousands_separator=','):
    print(con.sql("""
SELECT
    ROW_NUMBER() OVER (ORDER BY total_earned DESC) AS rank,
    vendor_name,
    total_contracts,
    total_earned
FROM (
SELECT
    vendor_name,
    COUNT(*) AS total_contracts,
    SUM(contract_value) AS total_earned
    FROM contracts_latest
    GROUP BY vendor_name
    ORDER BY total_earned DESC
    ) t
""").pl().limit(10))



shape: (10, 4)
┌──────┬─────────────────────────────────┬─────────────────┬───────────────────┐
│ rank ┆ vendor_name                     ┆ total_contracts ┆ total_earned      │
│ ---  ┆ ---                             ┆ ---             ┆ ---               │
│ i64  ┆ str                             ┆ i64             ┆ f64               │
╞══════╪═════════════════════════════════╪═════════════════╪═══════════════════╡
│ 1    ┆ Irving Shipbuilding Inc         ┆ 4               ┆ 20,402,436,551.91 │
│ 2    ┆ BANCTEC (CANADA), INC.          ┆ 8               ┆ 20,031,143,258.03 │
│ 3    ┆ General Dynamics Land Systems … ┆ 54              ┆ 8,625,542,848.69  │
│ 4    ┆ PCL CONSTRUCTORS CANADA INC.    ┆ 40              ┆ 8,314,979,097.55  │
│ 5    ┆ I.M.P Group Limited             ┆ 3               ┆ 8,166,089,765.70  │
│ 6    ┆ Irving Shipbuilding Inc.        ┆ 1               ┆ 8,010,267,025.00  │
│ 7    ┆ Sikorsky International Operati… ┆ 7               ┆ 7,607,385,308.00  │
│ 8    ┆ Casc